[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb)

# 🔴 Hard: Flash Attention (Tiled)

Реализуйте **tiled attention с online softmax** — ключевую алгоритмическую идею FlashAttention. Результат должен совпадать с обычным attention, но полная матрица `(S,S)` не должна храниться целиком.

## Почему это быстрее на GPU

FlashAttention не уменьшает асимптотику арифметики: сравнений `QKᵀ` по-прежнему $O(S^2D)$. Выигрыш появляется за счёт **IO-awareness**: маленькие блоки Q/K/V и промежуточное состояние удерживаются в быстрой памяти, а огромная матрица scores не записывается и не читается из HBM. В этом CPU-упражнении важен сам точный tiled-алгоритм, а не обещание ускорения.

## Online softmax

Обычный устойчивый softmax строки использует её максимум. При обработке K/V блоками будущий глобальный максимум пока неизвестен, поэтому для каждой строки Q хранят три состояния:

- `m` — текущий максимум обработанных scores;
- `l` — текущая сумма экспонент в системе координат максимума `m`;
- `acc` — взвешенная сумма V с тем же масштабом.

Для нового блока найдите его максимум и объединённый `m_new`. Старые `l` и `acc` нужно домножить на `exp(m_old - m_new)`, а вклад нового блока — на экспоненты относительно `m_new`. Это не приближение: в пределах погрешности floating point результат идентичен полному softmax.

### Пример разбиения

При `S=70` и `block_size=32` границы блоков равны `[0:32]`, `[32:64]`, `[64:70]`. Последний неполный блок обязателен. Для каждого Q-блока нужно пройти все три K/V-блока; состояние имеет формы примерно `(B, B_q, 1)` для `m,l` и `(B,B_q,D)` для аккумулятора.

### Signature
```python
def flash_attention(Q, K, V, block_size=32) -> Tensor:
    # Q, K, V: (B, S, D)
    # Returns: (B, S, D) — same as standard attention
```

### Key Insight
Instead of materializing the full S×S attention matrix, process in blocks:
1. For each Q-block, iterate over K/V blocks
2. Use **online softmax**: track running `max` and `sum`
3. Rescale accumulator when max changes: `acc *= exp(old_max - new_max)`
4. Final: `output = acc / row_sum`

Must give **identical** results to standard softmax attention.

## План реализации

1. Создайте выход и обходите Q диапазонами длины `block_size`.
2. Для каждого Q-блока инициализируйте `m=-inf`, `l=0`, `acc=0`.
3. Последовательно обработайте все K/V-блоки, обновляя состояния по формулам online softmax.
4. После последнего K/V-блока нормализуйте `acc / l` и запишите Q-часть выхода.

## Быстрые проверки

- Сравните с обычным scaled dot-product attention через `torch.allclose`.
- Проверьте `block_size=1`, `block_size>=S` и длину, не кратную размеру блока.
- Результат не должен зависеть от выбранного `block_size` сверх обычной погрешности.
- Backward по сумме outputs должен дать градиенты для Q, K, V.

## Частые ошибки

- Перемасштабировать `l`, но забыть перемасштабировать `acc`.
- Использовать максимум только нового блока вместо максимума старого и нового состояний.
- Потерять последнюю неполную плитку.
- Случайно материализовать глобальную `(S,S)` матрицу вне циклов.

## Материалы для глубокого изучения

- [FlashAttention: Fast and Memory-Efficient Exact Attention](https://arxiv.org/abs/2205.14135) — исходная статья, online softmax и IO complexity.
- [Official FlashAttention repository](https://github.com/Dao-AILab/flash-attention) — CUDA/Triton реализации и поддерживаемые варианты.
- [PyTorch SDPA tutorial](https://docs.pytorch.org/tutorials/intermediate/scaled_dot_product_attention_tutorial.html) — выбор Flash/Efficient/Math backend в PyTorch.

## Вопросы для самопроверки

1. Почему изменение running maximum требует пересчитать масштаб старого аккумулятора?
2. Что именно экономит FlashAttention: FLOPs или обращения к памяти?
3. Почему ответ должен быть почти независим от размера блока?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import math

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def flash_attention(Q, K, V, block_size=32):
    # Process Q in blocks, iterate K/V blocks with online softmax
    pass

In [ ]:
# 🧪 Debug
import math
Q, K, V = torch.randn(1, 8, 4), torch.randn(1, 8, 4), torch.randn(1, 8, 4)
out = flash_attention(Q, K, V, block_size=4)
scores = torch.bmm(Q, K.transpose(1,2)) / math.sqrt(4)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
print('Match:', torch.allclose(out, ref, atol=1e-4))

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('flash_attention')